In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
mobility_df = pd.read_csv("mobility_pings_featured.csv")
stores_df = pd.read_csv("stores.csv")

print("Mobility data shape:", mobility_df.shape)
print("Stores data shape:", stores_df.shape)

print("\nMobility columns:")
print(mobility_df.columns.tolist())

print("\nStore columns:")
print(stores_df.columns.tolist())

Mobility data shape: (5000, 16)
Stores data shape: (10, 7)

Mobility columns:
['ping_id', 'anonymous_user_id', 'timestamp', 'latitude', 'longitude', 'device_type', 'movement_type', 'date', 'hour', 'day_of_week', 'time_period', 'is_peak_hour', 'is_weekend', 'month', 'week_of_month', 'movement_type_code']

Store columns:
['store_id', 'store_name', 'latitude', 'longitude', 'store_type', 'daily_capacity', 'opening_date']


In [3]:
print("Mobility coordinates:")
print(mobility_df[["latitude", "longitude"]].head())

print("\nStore coordinates:")
print(stores_df[["latitude", "longitude"]].head())

Mobility coordinates:
    latitude  longitude
0  28.616127  77.197509
1  28.637518  77.208469
2  28.613040  77.234882
3  28.635834  77.195544
4  28.581480  77.237832

Store coordinates:
   latitude  longitude
0   28.6205     77.210
1   28.6300     77.220
2   28.6050     77.205
3   28.6250     77.230
4   28.6150     77.195


In [5]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [6]:
test_distance = haversine_distance(
    mobility_df.loc[0, "latitude"],
    mobility_df.loc[0, "longitude"],
    stores_df.loc[0, "latitude"],
    stores_df.loc[0, "longitude"]
)

print(f"Distance from first mobility ping to first store: {test_distance:.3f} km")

Distance from first mobility ping to first store: 1.313 km


In [7]:
def find_nearest_store(row):
    distances = stores_df.apply(
        lambda store: haversine_distance(
            row["latitude"],
            row["longitude"],
            store["latitude"],
            store["longitude"]
        ),
        axis=1
    )

    nearest_index = distances.idxmin()

    return pd.Series({
        "nearest_store_id": stores_df.loc[nearest_index, "store_id"],
        "nearest_store_name": stores_df.loc[nearest_index, "store_name"],
        "distance_to_nearest_store_km": distances.loc[nearest_index]
    })


nearest_store_data = mobility_df.apply(find_nearest_store, axis=1)

mobility_spatial = pd.concat(
    [mobility_df, nearest_store_data],
    axis=1
)

print("Spatial mapping completed.")
print("Rows:", len(mobility_spatial))

Spatial mapping completed.
Rows: 5000


In [8]:
mobility_spatial[
    [
        "ping_id",
        "latitude",
        "longitude",
        "nearest_store_id",
        "nearest_store_name",
        "distance_to_nearest_store_km"
    ]
].head(10)

,ping_id,latitude,longitude,nearest_store_id,nearest_store_name,distance_to_nearest_store_km
0,P00001,28.616127,77.197509,S005,GeoCafe Downtown,0.275110
1,P00002,28.637518,77.208469,S006,GeoCafe Mall,0.694555
2,P00003,28.613040,77.234882,S007,GeoCafe University,1.022159
3,P00004,28.635834,77.195544,S008,GeoCafe TechPark,0.106859
4,P00005,28.581480,77.237832,S010,GeoCafe South,1.982183
5,P00006,28.616687,77.233479,S004,GeoCafe Station,0.984764
6,P00007,28.621972,77.183217,S005,GeoCafe Downtown,1.387024
7,P00008,28.620578,77.238071,S004,GeoCafe Station,0.928638
8,P00009,28.578677,77.202910,S010,GeoCafe South,2.090403
9,P00010,28.650078,77.197526,S009,GeoCafe North,1.217253


In [9]:
def classify_distance(distance):
    if distance <= 0.5:
        return "Within 500m"
    elif distance <= 1.0:
        return "500m-1km"
    elif distance <= 2.0:
        return "1-2km"
    else:
        return "Above 2km"


mobility_spatial["distance_band"] = (
    mobility_spatial["distance_to_nearest_store_km"]
    .apply(classify_distance)
)

print(
    mobility_spatial["distance_band"]
    .value_counts()
)

distance_band
Above 2km      1485
500m-1km       1449
1-2km          1347
Within 500m     719
Name: count, dtype: int64


In [10]:
mobility_spatial["within_500m"] = (
    mobility_spatial["distance_to_nearest_store_km"] <= 0.5
).astype(int)

mobility_spatial["within_1km"] = (
    mobility_spatial["distance_to_nearest_store_km"] <= 1.0
).astype(int)

mobility_spatial[
    [
        "ping_id",
        "distance_to_nearest_store_km",
        "distance_band",
        "within_500m",
        "within_1km"
    ]
].head()

,ping_id,distance_to_nearest_store_km,distance_band,within_500m,within_1km
0,P00001,0.275110,Within 500m,1,1
1,P00002,0.694555,500m-1km,0,1
2,P00003,1.022159,1-2km,0,0
3,P00004,0.106859,Within 500m,1,1
4,P00005,1.982183,1-2km,0,0


In [11]:
store_analysis = (
    mobility_spatial
    .groupby(
        [
            "nearest_store_id",
            "nearest_store_name"
        ]
    )
    .agg(
        mobility_pings=("ping_id", "count"),
        avg_distance_km=("distance_to_nearest_store_km", "mean"),
        mobility_within_500m=("within_500m", "sum"),
        mobility_within_1km=("within_1km", "sum")
    )
    .reset_index()
)

store_analysis["catchment_500m_pct"] = (
    store_analysis["mobility_within_500m"]
    / store_analysis["mobility_pings"]
    * 100
)

store_analysis["catchment_1km_pct"] = (
    store_analysis["mobility_within_1km"]
    / store_analysis["mobility_pings"]
    * 100
)

store_analysis = store_analysis.round(2)

store_analysis

,nearest_store_id,nearest_store_name,mobility_pings,avg_distance_km,mobility_within_500m,mobility_within_1km,catchment_500m_pct,catchment_1km_pct
0,S001,GeoCafe Central,316,0.63,101,288,31.96,91.14
1,S002,GeoCafe Market,199,0.65,64,174,32.16,87.44
2,S003,GeoCafe Riverside,536,1.48,89,269,16.60,50.19
3,S004,GeoCafe Station,602,1.91,68,193,11.30,32.06
4,S005,GeoCafe Downtown,702,1.85,76,242,10.83,34.47
5,S006,GeoCafe Mall,283,1.17,69,162,24.38,57.24
6,S007,GeoCafe University,517,1.24,91,299,17.60,57.83
7,S008,GeoCafe TechPark,648,1.74,68,235,10.49,36.27
8,S009,GeoCafe North,513,1.97,50,147,9.75,28.65
9,S010,GeoCafe South,684,2.02,43,159,6.29,23.25


In [12]:
store_analysis_sorted = store_analysis.sort_values(
    "mobility_pings",
    ascending=False
)

store_analysis_sorted

,nearest_store_id,nearest_store_name,mobility_pings,avg_distance_km,mobility_within_500m,mobility_within_1km,catchment_500m_pct,catchment_1km_pct
4,S005,GeoCafe Downtown,702,1.85,76,242,10.83,34.47
9,S010,GeoCafe South,684,2.02,43,159,6.29,23.25
7,S008,GeoCafe TechPark,648,1.74,68,235,10.49,36.27
3,S004,GeoCafe Station,602,1.91,68,193,11.30,32.06
2,S003,GeoCafe Riverside,536,1.48,89,269,16.60,50.19
6,S007,GeoCafe University,517,1.24,91,299,17.60,57.83
8,S009,GeoCafe North,513,1.97,50,147,9.75,28.65
0,S001,GeoCafe Central,316,0.63,101,288,31.96,91.14
5,S006,GeoCafe Mall,283,1.17,69,162,24.38,57.24
1,S002,GeoCafe Market,199,0.65,64,174,32.16,87.44


In [13]:
movement_distance_analysis = (
    mobility_spatial
    .groupby(
        ["distance_band", "movement_type"]
    )
    .size()
    .reset_index(name="mobility_pings")
    .sort_values(
        ["distance_band", "mobility_pings"],
        ascending=[True, False]
    )
)

movement_distance_analysis

,distance_band,movement_type,mobility_pings
2,1-2km,Walking,643
0,1-2km,Driving,413
1,1-2km,Public Transport,291
5,500m-1km,Walking,705
3,500m-1km,Driving,459
4,500m-1km,Public Transport,285
8,Above 2km,Walking,737
6,Above 2km,Driving,452
7,Above 2km,Public Transport,296
11,Within 500m,Walking,348


In [14]:
time_catchment_analysis = (
    mobility_spatial
    .groupby("time_period")
    .agg(
        total_pings=("ping_id", "count"),
        within_500m=("within_500m", "sum"),
        within_1km=("within_1km", "sum")
    )
    .reset_index()
)

time_catchment_analysis["within_500m_pct"] = (
    time_catchment_analysis["within_500m"]
    / time_catchment_analysis["total_pings"]
    * 100
)

time_catchment_analysis["within_1km_pct"] = (
    time_catchment_analysis["within_1km"]
    / time_catchment_analysis["total_pings"]
    * 100
)

time_catchment_analysis.round(2)

,time_period,total_pings,within_500m,within_1km,within_500m_pct,within_1km_pct
0,Afternoon,1055,166,455,15.73,43.13
1,Evening,821,111,345,13.52,42.02
2,Morning,1462,198,629,13.54,43.02
3,Night,1662,244,739,14.68,44.46


In [15]:
store_capacity_analysis = store_analysis.merge(
    stores_df[
        [
            "store_id",
            "store_name",
            "store_type",
            "daily_capacity",
            "opening_date"
        ]
    ],
    left_on="nearest_store_id",
    right_on="store_id",
    how="left",
    suffixes=("", "_store")
)

store_capacity_analysis

,nearest_store_id,nearest_store_name,mobility_pings,avg_distance_km,mobility_within_500m,mobility_within_1km,catchment_500m_pct,catchment_1km_pct,store_id,store_name,store_type,daily_capacity,opening_date
0,S001,GeoCafe Central,316,0.63,101,288,31.96,91.14,S001,GeoCafe Central,High Street,450,2024-01-15
1,S002,GeoCafe Market,199,0.65,64,174,32.16,87.44,S002,GeoCafe Market,Market,400,2024-03-10
2,S003,GeoCafe Riverside,536,1.48,89,269,16.60,50.19,S003,GeoCafe Riverside,Riverside,350,2024-05-20
3,S004,GeoCafe Station,602,1.91,68,193,11.30,32.06,S004,GeoCafe Station,Transit,500,2024-07-01
4,S005,GeoCafe Downtown,702,1.85,76,242,10.83,34.47,S005,GeoCafe Downtown,Downtown,550,2024-09-15
5,S006,GeoCafe Mall,283,1.17,69,162,24.38,57.24,S006,GeoCafe Mall,Shopping Mall,700,2025-01-10
6,S007,GeoCafe University,517,1.24,91,299,17.60,57.83,S007,GeoCafe University,University,300,2025-02-15
7,S008,GeoCafe TechPark,648,1.74,68,235,10.49,36.27,S008,GeoCafe TechPark,Tech Park,600,2025-04-20
8,S009,GeoCafe North,513,1.97,50,147,9.75,28.65,S009,GeoCafe North,High Street,400,2025-06-10
9,S010,GeoCafe South,684,2.02,43,159,6.29,23.25,S010,GeoCafe South,Residential,350,2025-08-01


In [16]:
mobility_spatial.to_csv(
    "mobility_store_proximity.csv",
    index=False
)

print("Saved: mobility_store_proximity.csv")

Saved: mobility_store_proximity.csv


In [17]:
store_capacity_analysis.to_csv(
    "store_spatial_analysis.csv",
    index=False
)

print("Saved: store_spatial_analysis.csv")

Saved: store_spatial_analysis.csv


In [18]:
print("========== DAY 5 VALIDATION ==========")

print("Mobility records:", len(mobility_spatial))
print("Stores:", len(stores_df))

print(
    "Average nearest-store distance:",
    round(
        mobility_spatial["distance_to_nearest_store_km"].mean(),
        3
    ),
    "km"
)

print(
    "Pings within 500m:",
    mobility_spatial["within_500m"].sum()
)

print(
    "Pings within 1km:",
    mobility_spatial["within_1km"].sum()
)

print("\nFiles created:")
print("- mobility_store_proximity.csv")
print("- store_spatial_analysis.csv")

========== DAY 5 VALIDATION ==========
Mobility records: 5000
Stores: 10
Average nearest-store distance: 1.613 km
Pings within 500m: 719
Pings within 1km: 2168

Files created:
- mobility_store_proximity.csv
- store_spatial_analysis.csv


## Day 5 — Spatial Analytics with Python

- Performed geographic distance calculations using latitude and longitude
- Implemented Haversine distance calculation
- Mapped each mobility ping to its nearest retail store
- Calculated distance between mobility points and nearest stores
- Created distance bands for spatial proximity analysis
- Created 500m and 1km catchment indicators
- Performed store-level mobility analysis
- Analyzed mobility by movement type and distance band
- Analyzed catchment coverage across different time periods
- Combined spatial mobility metrics with store information
- Generated spatial analytics datasets for downstream analysis